# Fine-tuning Qwen3-VL-2B-Instruct su IU X-Ray (dataset tradotto in italiano)

Questo notebook guida il processo completo di fine-tuning QLoRA del modello Qwen3-VL-2B-Instruct
sul dataset IU X-Ray (Indiana University Chest X-Rays) con referti tradotti in italiano.

**Obiettivo**: dato un'immagine RX del torace, generare automaticamente findings e impression in italiano.

---

## Sezioni del notebook

1. Setup ambiente e dipendenze
2. Verifica GPU
3. Configurazione iperparametri
4. Caricamento dataset (JSON pre-processati da `split_dataset.py`)
5. Caricamento modello e configurazione QLoRA
6. Training (dry run + training completo)
7. Valutazione (ROUGE + BERTScore + campioni qualitativi)
8. Inferenza su singola immagine

---

> **Prima di iniziare**: assicurati di aver eseguito `split_dataset.py` in locale
> e di aver caricato su RunPod la seguente struttura:
> ```
> /workspace/
>   data/
>     dataset_filtrato.csv
>     images/
>       images_normalized/      # tutti i .png
>     processed/
>       train.json
>       val.json
>       test.json
>       split_manifest.json
>   outputs/
>     checkpoints/
>     evaluation/
> ```

---
## 1. Setup ambiente e dipendenze

Esegui questa cella una sola volta. Installa tutte le librerie necessarie.
Richiede qualche minuto.

In [1]:
import sys
print(f"Python: {sys.executable}")
# Deve stampare un path che contiene .venv — se vedi il Python di sistema c'è un problema

import torch
import transformers
print(f"✅ Torch:        {torch.__version__}")
print(f"✅ Transformers: {transformers.__version__}")
print(f"✅ CUDA:         {torch.cuda.is_available()}")

Python: /mnt/maselli/develop/shield/.venv/bin/python
✅ Torch:        2.11.0+cu130
✅ Transformers: 5.3.0
✅ CUDA:         True


---
## 2. Verifica GPU

Verifica che la GPU sia rilevata correttamente e controlla la VRAM disponibile.

In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ Nessuna GPU disponibile. Verifica il setup CUDA su RunPod.")

n_gpus = torch.cuda.device_count()
print(f"GPU disponibili: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / 1e9
    print(f"  GPU {i}: {props.name} — {vram_gb:.1f} GB VRAM")

import transformers, peft, trl
print(f"\nVersioni librerie:")
print(f"  PyTorch:       {torch.__version__}")
print(f"  Transformers:  {transformers.__version__}")
print(f"  PEFT:          {peft.__version__}")
print(f"  TRL:           {trl.__version__}")
print("\n✅ Setup GPU verificato!")

GPU disponibili: 1
  GPU 0: NVIDIA H200-141C — 150.9 GB VRAM

Versioni librerie:
  PyTorch:       2.11.0+cu130
  Transformers:  5.3.0
  PEFT:          0.18.1
  TRL:           0.29.1

✅ Setup GPU verificato!


---
## 3. Configurazione

Tutti gli iperparametri in un unico posto. Modifica qui prima di procedere.

I valori di default sono ottimizzati per una **RTX 4090 (24 GB VRAM)** con QLoRA.

**Nota sul learning rate**: si usa `1e-4` invece di `2e-4` per ridurre il rischio di
catastrofic forgetting sulle rappresentazioni visive con dataset medico piccolo (~3-4k campioni).

In [3]:
import random
import numpy as np
import torch
from pathlib import Path

# ── Riproducibilità ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Percorsi RunPod ──────────────────────────────────────────
# I JSON sono prodotti da split_dataset.py e caricati su RunPod prima di eseguire questo notebook.
DATA_DIR       = Path("/home/maselli/develop/shield/data")   # dove sono train/val/test.json
PROCESSED_DIR  = Path("/home/maselli/develop/shield/data/processed")   # dove sono train/val/test.json
IMAGES_DIR     = Path("/home/maselli/develop/shield/data/images/images_normalized")
CHECKPOINT_DIR = Path("/home/maselli/develop/shield/outputs/checkpoints")
EVAL_DIR       = Path("/home/maselli/develop/shield/outputs/evaluation")

for d in [PROCESSED_DIR, CHECKPOINT_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── System prompt ────────────────────────────────────────────
# Deve essere identico a quello usato in split_dataset.py per la coerenza train/inferenza.
SYSTEM_PROMPT = (
    "Sei un radiologo esperto. "
    "Ti verranno fornite una o più immagini mediche. "
    "Il tuo compito è: "
    "1) Descrivere in modo conciso i reperti visibili nell'immagine o nelle immagini, "
    "incluse le strutture anatomiche, le anomalie e le osservazioni rilevanti. "
    "2) Fornire un'impressione clinica concisa che riassuma i reperti principali. "
    "Fornisci la risposta in ESATTAMENTE questo formato:\n"
    "Reperti:\n<i tuoi reperti dettagliati qui>\n\n"
    "Impressione:\n<la tua impressione concisa qui>\n\n"
    "NON includere altre sezioni o preamboli."
)

# ── Modello ─────────────────────────────────────────────────
MODEL_NAME = "/home/maselli/develop/shield/models/Qwen3-VL-2B-Instruct"

# ── QLoRA ───────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32      # scala effettiva = lora_alpha / r = 2.0
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ── Training ────────────────────────────────────────────────
NUM_EPOCHS              = 3
BATCH_SIZE              = 4    
GRAD_ACCUMULATION_STEPS = 4  
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH          = 2048
MAX_IMAGE_PIXELS        = 1003520  # riduci a 501760 se vai OOM

# ── Wandb (opzionale) ────────────────────────────────────────
USE_WANDB     = False
WANDB_PROJECT = "xray-finetune-ita"

print("✅ Configurazione caricata!")
print(f"   Modello:         {MODEL_NAME}")
print(f"   LoRA rank:       {LORA_R}")
print(f"   Epoch:           {NUM_EPOCHS}")
print(f"   Effective batch: {BATCH_SIZE * GRAD_ACCUMULATION_STEPS}")
print(f"   Learning rate:   {LEARNING_RATE}")
print(f"   Seed:            {SEED}")
print(f"   Processed dir:   {PROCESSED_DIR}")


✅ Configurazione caricata!
   Modello:         /home/maselli/develop/shield/models/Qwen3-VL-2B-Instruct
   LoRA rank:       16
   Epoch:           3
   Effective batch: 16
   Learning rate:   0.0002
   Seed:            42
   Processed dir:   /home/maselli/develop/shield/data/processed


---
## 4. Caricamento dataset

Carica i JSON già processati da `split_dataset.py`.
Lo split è stato eseguito **per uid (paziente)** prima del caricamento:
nessun paziente appare in più di uno split (no data leakage).

Il formato di ogni esempio include:
- un messaggio `system` con il system prompt del radiologo
- un messaggio `user` con l'immagine RX
- un messaggio `assistant` con reperti e impressione nel formato atteso


In [4]:
import json
from pathlib import Path

# Verifica che i file esistano
train_path    = PROCESSED_DIR / "train.json"
val_path      = PROCESSED_DIR / "val.json"
test_path     = PROCESSED_DIR / "test.json"
manifest_path = PROCESSED_DIR / "split_manifest.json"

for p in [train_path, val_path, test_path, manifest_path, IMAGES_DIR]:
    status = "✅" if p.exists() else "❌ NON TROVATO"
    print(f"{status}  {p}")

if not all(p.exists() for p in [train_path, val_path, test_path]):
    raise FileNotFoundError(
        "JSON mancanti. Esegui split_dataset.py e carica i file in /workspace/data/processed/"
    )

# Mostra info dal manifest
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"\nManifest split:")
    print(f"  Seed:       {manifest['seed']}")
    print(f"  Train:      {manifest['train_size']:.0%}")
    print(f"  Val:        {manifest['val_size']:.0%}")
    print(f"  Test:       {manifest['test_size']:.0%}")
    print(f"  CSV sorgente: {manifest.get('csv_input', 'n/d')}")

# Conta esempi
with open(train_path) as f: n_train = len(json.load(f))
with open(val_path)   as f: n_val   = len(json.load(f))
with open(test_path)  as f: n_test  = len(json.load(f))

print(f"\nEsempi:")
print(f"  Train: {n_train}")
print(f"  Val:   {n_val}")
print(f"  Test:  {n_test}")
print(f"  Totale: {n_train + n_val + n_test}")

# Immagini disponibili
n_images = len(list(IMAGES_DIR.glob("*.png"))) if IMAGES_DIR.exists() else 0
print(f"\nImmagini su disco: {n_images}")


✅  /home/maselli/develop/shield/data/processed/train.json
✅  /home/maselli/develop/shield/data/processed/val.json
✅  /home/maselli/develop/shield/data/processed/test.json
✅  /home/maselli/develop/shield/data/processed/split_manifest.json
✅  /home/maselli/develop/shield/data/images/images_normalized

Manifest split:
  Seed:       42
  Train:      80%
  Val:        10%
  Test:       10%
  CSV sorgente: /home/maselli/develop/shield/data/dataset_filtrato.csv

Esempi:
  Train: 2554
  Val:   319
  Test:  320
  Totale: 3193

Immagini su disco: 7470


In [5]:
import json
from datasets import Dataset

def load_json_dataset(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Normalizza: content assistant da stringa a lista
    for ex in data:
        for msg in ex["messages"]:
            if isinstance(msg["content"], str):
                msg["content"] = [{"type": "text", "text": msg["content"]}]
    
    return Dataset.from_list(data)

train_dataset = load_json_dataset(PROCESSED_DIR / "train.json")
val_dataset   = load_json_dataset(PROCESSED_DIR / "val.json")

# Tieni test_data come lista — serve così per la valutazione
with open(PROCESSED_DIR / "test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"✅ Dataset caricato!")
print(f"   Train: {len(train_dataset)} esempi")
print(f"   Val:   {len(val_dataset)} esempi")
print(f"   Test:  {len(test_data)} esempi")

# ── Anteprima primo esempio ───────────────────────────────────
print("\n" + "=" * 60)
print(" ESEMPIO DI TRAINING (formato chat-template)")
print("=" * 60)
example = train_dataset[0]
for msg in example["messages"]:
    role    = msg["role"].upper()
    content = msg["content"]
    if isinstance(content, list):
        content = str(content)
    content_str = str(content)
    print(f"\n[{role}]")
    print(content_str[:300] + "..." if len(content_str) > 300 else content_str)
print("\n" + "=" * 60)


✅ Dataset caricato!
   Train: 2554 esempi
   Val:   319 esempi
   Test:  320 esempi

 ESEMPIO DI TRAINING (formato chat-template)

[SYSTEM]
[{'image': None, 'text': "Sei un radiologo esperto. Ti verranno fornite una o più immagini mediche. Il tuo compito è: 1) Descrivere in modo conciso i reperti visibili nell'immagine o nelle immagini, incluse le strutture anatomiche, le anomalie e le osservazioni rilevanti. 2) Fornire un'impressione c...

[USER]
[{'image': '/home/maselli/develop/shield/data/images/images_normalized/2_IM-0652-1001.dcm.png', 'text': None, 'type': 'image'}]

[ASSISTANT]
[{'image': None, 'text': 'Reperti:\nCardiomegalia borderline. Sternotomia mediana XXXX. Arterie polmonari dilatate. Polmoni liberi. Inferiore XXXX XXXX XXXX.\n\nImpressione:\nNessun reperto polmonare acuto.', 'type': 'text'}]



In [6]:
# ── Diagnostica lunghezza risposte ───────────────────────────
# Conta le parole nella risposta dell'assistant per stimare
# il rischio di troncamento a MAX_SEQ_LENGTH token.
print("\n" + "=" * 60)
print(" DIAGNOSTICA DATASET")
print("=" * 60)

def extract_assistant_text(example):
    for msg in example["messages"]:
        if msg["role"] == "assistant":
            content = msg["content"]
            if isinstance(content, str):
                return content
            elif isinstance(content, list):
                # estrai il testo dal primo elemento di tipo text
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "text":
                        return item["text"]
    return ""

all_responses = [extract_assistant_text(ex) for ex in train_dataset]
response_lens = [len(r.split()) for r in all_responses if r]

import statistics
print(f"\nLunghezza risposta assistant (parole) — train set:")
print(f"  min:     {min(response_lens)}")
print(f"  max:     {max(response_lens)}")
print(f"  media:   {statistics.mean(response_lens):.0f}")
print(f"  mediana: {statistics.median(response_lens):.0f}")

soglia = 300
lunghi = sum(1 for l in response_lens if l > soglia)
if lunghi > 0:
    print(f"\n⚠️  Risposte con >{soglia} parole (rischio troncamento): {lunghi}")
else:
    print(f"\n✅ Nessuna risposta supera {soglia} parole")

# Verifica coerenza system prompt nei JSON
sample = train_dataset[0]
system_msgs = [m for m in sample["messages"] if m["role"] == "system"]
if system_msgs:
    print(f"\n✅ System prompt presente negli esempi")
    print(f"   Primi 80 caratteri: {system_msgs[0]['content'][:80]}...")
else:
    print(f"\n⚠️  System prompt NON trovato negli esempi — verifica split_dataset.py")



 DIAGNOSTICA DATASET

Lunghezza risposta assistant (parole) — train set:
  min:     13
  max:     1037
  media:   45
  mediana: 40

⚠️  Risposte con >300 parole (rischio troncamento): 2

✅ System prompt presente negli esempi
   Primi 80 caratteri: [{'image': None, 'text': "Sei un radiologo esperto. Ti verranno fornite una o più immagini mediche. Il tuo compito è: 1) Descrivere in modo conciso i reperti visibili nell'immagine o nelle immagini, incluse le strutture anatomiche, le anomalie e le osservazioni rilevanti. 2) Fornire un'impressione clinica concisa che riassuma i reperti principali. Fornisci la risposta in ESATTAMENTE questo formato:\nReperti:\n<i tuoi reperti dettagliati qui>\n\nImpressione:\n<la tua impressione concisa qui>\n\nNON includere altre sezioni o preamboli.", 'type': 'text'}]...


In [7]:
# ── Verifica data leakage dal manifest ───────────────────────
# Lo split è già stato verificato da split_dataset.py.
# Qui confirmiamo leggendo il manifest salvato.

with open(PROCESSED_DIR / "split_manifest.json") as f:
    manifest = json.load(f)

train_set = set(manifest["train_uids"])
val_set   = set(manifest["val_uids"])
test_set  = set(manifest["test_uids"])

leak_tv = train_set & val_set
leak_tt = train_set & test_set
leak_vt = val_set   & test_set

if leak_tv or leak_tt or leak_vt:
    print("❌ DATA LEAKAGE RILEVATO nel manifest:")
    if leak_tv: print(f"   Train ∩ Val:  {len(leak_tv)} uid")
    if leak_tt: print(f"   Train ∩ Test: {len(leak_tt)} uid")
    if leak_vt: print(f"   Val ∩ Test:   {len(leak_vt)} uid")
    raise ValueError("Interrompi e rigenera lo split con split_dataset.py")
else:
    print(f"✅ Nessun data leakage — split per paziente verificato dal manifest")
    print(f"   Train: {len(train_set)} pazienti")
    print(f"   Val:   {len(val_set)} pazienti")
    print(f"   Test:  {len(test_set)} pazienti")


✅ Nessun data leakage — split per paziente verificato dal manifest
   Train: 2554 pazienti
   Val:   319 pazienti
   Test:  320 pazienti


---
## 5. Caricamento modello e configurazione QLoRA

Carica Qwen3-VL-2B-Instruct con quantizzazione 4-bit (QLoRA).
Al primo avvio scarica i pesi da HuggingFace (~5 GB).

**Nota**: il modello viene preparato per QLoRA **prima** di creare la LoraConfig,
e `peft_config` non viene passato allo `SFTTrainer` per evitare il double-wrapping.

In [8]:
import torch
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Quantizzazione 4-bit ─────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# ── Carica modello ────────────────────────────────────────────
print(f"Caricamento {MODEL_NAME}...")
print("(Al primo avvio scarica ~5 GB da HuggingFace — attendere)")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    #attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

# ── Carica processor ──────────────────────────────────────────
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=MAX_IMAGE_PIXELS,
)

print("\n✅ Modello caricato!")
vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"   VRAM usata: {vram_used:.1f} GB / {vram_total:.1f} GB")

Caricamento /home/maselli/develop/shield/models/Qwen3-VL-2B-Instruct...
(Al primo avvio scarica ~5 GB da HuggingFace — attendere)


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

/home/maselli/develop/shield/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



✅ Modello caricato!
   VRAM usata: 0.0 GB / 150.1 GB


In [9]:
# ── Prepara modello per QLoRA ────────────────────────────────
model = prepare_model_for_kbit_training(model)

# ── Configura LoRA — solo sul decoder linguistico ─────────────
# FIX: usiamo layers_to_transform per escludere il vision encoder.
# In Qwen3-VL il LLM decoder occupa model.layers[0..N-1].
# Recuperiamo dinamicamente il numero di layer del LLM.
n_llm_layers = model.config.text_config.num_hidden_layers
print(f"Layer LLM decoder: {n_llm_layers}")

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
    # Applica LoRA solo ai layer del LLM, non al vision encoder
    layers_to_transform=list(range(n_llm_layers)),
    layers_pattern="layers",
)

# FIX: il modello viene wrappato QUI manualmente.
# NON passiamo peft_config allo SFTTrainer per evitare double-wrapping.
model = get_peft_model(model, peft_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print("=" * 50)
print(f"Parametri trainable:  {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Parametri totali:     {total:,}")
print("=" * 50)
print("\n✅ LoRA configurato (solo decoder LLM)!")

Layer LLM decoder: 28
Parametri trainable:  17,432,576 (1.41%)
Parametri totali:     1,238,994,944

✅ LoRA configurato (solo decoder LLM)!


---
## 6. Training

Prima esegui il **dry run** (cella 6a) per verificare che la pipeline funzioni senza errori.
Solo se il dry run va a buon fine, lancia il training completo (cella 6b).

### Note metodologiche sul data collator

Il `XRayDataCollator` corregge tre bug presenti nella versione precedente:

1. **Label masking corretto per sequenze multimodali**: il confine prompt/risposta
   viene trovato cercando la posizione del token `<|im_start|>assistant` direttamente
   in `input_ids`, non stimando un offset dalla tokenizzazione del solo testo.
   Con Qwen3-VL le image tokens vengono inserite dal processor e sfasano qualsiasi
   conteggio basato sul testo.

2. **Padding token mascherati**: i token di padding vengono impostati a `-100`
   per escluderli dalla loss.

3. **Fallback con warning**: se il token `<|im_start|>assistant` non viene trovato
   (anomalia di formato), viene loggato un warning invece di mascherare silenziosamente
   la sequenza intera.

In [10]:
import os
import warnings
from PIL import Image
from transformers import TrainingArguments
from trl import SFTTrainer


class XRayDataCollator:
    """
    Collator custom per Qwen3-VL con system prompt e dataset tradotto in italiano.

    Label masking:
    - Maschera system prompt + messaggio utente + image tokens con -100.
    - Trova il confine cercando il token <|im_start|>assistant direttamente
      in input_ids — robusto alle image tokens che sfasano qualsiasi offset testuale.
    - Maschera i token di padding con -100.
    """

    def __init__(self, processor, max_seq_length: int = 2048):
        self.processor      = processor
        self.max_seq_length = max_seq_length
        self._im_start_id   = processor.tokenizer.convert_tokens_to_ids("<|im_start|>")
        self._assistant_ids = processor.tokenizer(
            "assistant", add_special_tokens=False
        )["input_ids"]

    def _find_assistant_start(self, input_ids_row):
        """
        Trova l'indice del primo token DOPO '<|im_start|>assistant\n' nell'ultima
        occorrenza (= turno dell'assistant nel nostro esempio single-turn).
        """
        ids     = input_ids_row.tolist()
        pattern = [self._im_start_id] + self._assistant_ids
        plen    = len(pattern)
        for i in range(len(ids) - plen, -1, -1):
            if ids[i:i + plen] == pattern:
                return i + plen + 1   # +1 salta il newline dopo 'assistant'
        return None

    def __call__(self, examples):
        texts       = []
        images_list = []

        for example in examples:
            messages = example["messages"]

            # Estrai immagine dal messaggio utente
            image_path = None
            for msg in messages:
                if msg["role"] == "user":
                    for item in msg["content"]:
                        if isinstance(item, dict) and item.get("type") == "image":
                            image_path = item["image"]

            if image_path and os.path.exists(image_path):
                img = Image.open(image_path).convert("RGB")
            else:
                warnings.warn(f"Immagine non trovata: {image_path} — usando placeholder grigio")
                img = Image.new("RGB", (224, 224), color=(128, 128, 128))
            images_list.append([img])

            text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
            )
            texts.append(text)

        batch = self.processor(
            text=texts,
            images=images_list,
            padding=True,
            truncation=True,
            max_length=self.max_seq_length,
            return_tensors="pt",
        )

        # Label masking — maschera tutto tranne la risposta dell'assistant
        labels = batch["input_ids"].clone()
        for i in range(labels.shape[0]):
            assistant_start = self._find_assistant_start(labels[i])
            if assistant_start is None:
                warnings.warn(
                    f"Esempio {i}: token '<|im_start|>assistant' non trovato. "
                    "Maschero l'intera sequenza — l'esempio non contribuirà alla loss."
                )
                labels[i, :] = -100
            else:
                labels[i, :assistant_start] = -100

        # Maschera padding
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch


# ── Inizializza collator ──────────────────────────────────────
# train_dataset e val_dataset sono già caricati nella cella 9
data_collator = XRayDataCollator(processor, max_seq_length=MAX_SEQ_LENGTH)

print(f"✅ Data collator pronto!")
print(f"   Train: {len(train_dataset)} esempi | Val: {len(val_dataset)} esempi")


✅ Data collator pronto!
   Train: 2554 esempi | Val: 319 esempi


In [11]:
# ── Verifica mascheratura label su un batch reale ─────────────
# Questo test è fondamentale: conferma che il collator maschera correttamente
# prompt + image tokens e lascia unmasked solo i token della risposta.
print("\n" + "=" * 60)
print(" VERIFICA MASCHERATURA LABEL")
print("=" * 60)

sample_batch = data_collator([train_dataset[i] for i in range(2)])
labels    = sample_batch["labels"]
input_ids = sample_batch["input_ids"]
attn_mask = sample_batch["attention_mask"]

for i in range(len(labels)):
    total_tokens    = attn_mask[i].sum().item()  # token reali (no padding)
    masked_tokens   = (labels[i] == -100).sum().item()
    unmasked_tokens = (labels[i] != -100).sum().item()
    # I pad token vanno in -100 quindi masked_tokens può superare total_tokens
    response_tokens = unmasked_tokens  # questi sono sicuramente sulla risposta
    pct_response    = 100 * response_tokens / total_tokens if total_tokens > 0 else 0

    print(f"\n  Esempio {i}:")
    print(f"    Token reali (no padding):          {total_tokens}")
    print(f"    Token mascherati (prompt+img+pad): {masked_tokens}")
    print(f"    Token risposta (loss attiva):      {response_tokens} ({pct_response:.1f}% dei reali)")

    if response_tokens == 0:
        print(f"    ❌ ERRORE: nessun token contribuisce alla loss — bug nel collator")
    elif pct_response < 3:
        print(f"    ⚠️  Meno del 3% dei token è risposta — verifica il collator")
    elif masked_tokens == 0:
        print(f"    ⚠️  Nessun token mascherato — il modello impara anche sul prompt")
    else:
        print(f"    ✅ Mascheratura corretta")

    # Sanity check: decodifica i token non mascherati — devono essere REPERTI/CONCLUSIONI
    unmasked_ids = input_ids[i][labels[i] != -100]
    decoded = processor.tokenizer.decode(unmasked_ids[:50], skip_special_tokens=True)
    print(f"    Primi token risposta (decodificati): {repr(decoded[:120])}")

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n  VRAM attuale: {vram_used:.1f} GB / {vram_total:.1f} GB")


 VERIFICA MASCHERATURA LABEL

  Esempio 0:
    Token reali (no padding):          1187
    Token mascherati (prompt+img+pad): 1329
    Token risposta (loss attiva):      61 (5.1% dei reali)
    ✅ Mascheratura corretta
    Primi token risposta (decodificati): 'Reperti:\nCardiomegalia borderline. Sternotomia mediana XXXX. Arterie polmonari dilatate. Polmoni liberi. Inferiore XXXX '

  Esempio 1:
    Token reali (no padding):          1390
    Token mascherati (prompt+img+pad): 1117
    Token risposta (loss attiva):      273 (19.6% dei reali)
    ✅ Mascheratura corretta
    Primi token risposta (decodificati): 'Reperti:\nSono presenti opacità interstiziali e alveolari bilaterali diffuse, compatibili con malattia polmonare ostrutti'

  VRAM attuale: 0.0 GB / 150.1 GB


In [12]:
# ── 6a. DRY RUN ───────────────────────────────────────────────
from trl import SFTConfig

print("🔍 DRY RUN — 10 esempi, 1 epoch")
print("Questo non produce un modello utile, serve solo a verificare la pipeline.\n")

dry_train = train_dataset.select(range(10))
dry_val   = val_dataset.select(range(5))

dry_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR / "dry_run"),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=LEARNING_RATE,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=1,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

dry_trainer = SFTTrainer(
    model=model,
    args=dry_args,
    train_dataset=dry_train,
    eval_dataset=dry_val,
    data_collator=data_collator,
)

dry_trainer.train()

# ── Diagnostica post dry run ──────────────────────────────────
print("\n" + "=" * 60)
print(" DIAGNOSTICA DRY RUN")
print("=" * 60)

log_history  = dry_trainer.state.log_history
train_losses = [x["loss"]      for x in log_history if "loss"      in x and "eval_loss" not in x]
eval_losses  = [x["eval_loss"] for x in log_history if "eval_loss" in x]

if train_losses:
    print(f"\n  Loss training steps: {[round(l, 4) for l in train_losses]}")
    if train_losses[-1] < 0.01:
        print("  ⚠️  Loss troppo bassa — possibile bug nella mascheratura delle label")
    elif train_losses[0] > 10:
        print("  ⚠️  Loss iniziale molto alta — verifica il formato del dataset")
    else:
        print("  ✅ Loss in range normale")

if eval_losses:
    print(f"\n  Loss validazione: {[round(l, 4) for l in eval_losses]}")

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n  VRAM dopo dry run: {vram_used:.1f} GB / {vram_total:.1f} GB")
print(f"  VRAM disponibile:  {vram_total - vram_used:.1f} GB")

if vram_total - vram_used < 2:
    print("  ⚠️  Poco margine VRAM — rischio OOM nel training completo")
    print("      Prova ad aumentare GRAD_ACCUMULATION_STEPS o ridurre MAX_IMAGE_PIXELS")
else:
    print("  ✅ Margine VRAM sufficiente per il training completo")

print("\n✅ Dry run completato senza errori!")
print("Puoi procedere con il training completo nella cella successiva.")

🔍 DRY RUN — 10 esempi, 1 epoch
Questo non produce un modello utile, serve solo a verificare la pipeline.



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,2.027255,1.852935



 DIAGNOSTICA DRY RUN

  Loss training steps: [2.8235, 2.0558, 2.5029, 1.9269, 2.0273]
  ✅ Loss in range normale

  Loss validazione: [1.8529]

  VRAM dopo dry run: 0.0 GB / 150.1 GB
  VRAM disponibile:  150.1 GB
  ✅ Margine VRAM sufficiente per il training completo

✅ Dry run completato senza errori!
Puoi procedere con il training completo nella cella successiva.


In [13]:
# ── 6b. TRAINING COMPLETO ─────────────────────────────────────
from trl import SFTConfig

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name="qwen3vl-xray-qlora-ita")

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=True,
    fp16=False,
    save_strategy="epoch",
    save_total_limit=3,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    report_to="wandb" if USE_WANDB else "none",
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("🚀 Avvio training completo...")
print(f"   Epoch:           {NUM_EPOCHS}")
print(f"   Esempi train:    {len(train_dataset)}")
print(f"   Effective batch: {BATCH_SIZE * GRAD_ACCUMULATION_STEPS}")
print(f"   Checkpoint ogni: fine epoch")
print()

trainer.train()

# ── Diagnostica post training ─────────────────────────────────
print("\n" + "=" * 60)
print(" DIAGNOSTICA TRAINING")
print("=" * 60)

log_history  = trainer.state.log_history
train_losses = [x["loss"]      for x in log_history if "loss"      in x and "eval_loss" not in x]
eval_losses  = [x["eval_loss"] for x in log_history if "eval_loss" in x]

if train_losses:
    print(f"\n  Loss training — inizio: {train_losses[0]:.4f}  |  fine: {train_losses[-1]:.4f}")
    if train_losses[-1] >= train_losses[0]:
        print("  ⚠️  La loss non è scesa — verifica learning rate e dataset")
    elif train_losses[-1] < 0.01:
        print("  ⚠️  Loss finale molto bassa — possibile overfitting")
    else:
        print("  ✅ Loss scesa correttamente")

if eval_losses:
    print(f"\n  Loss validazione — inizio: {eval_losses[0]:.4f}  |  fine: {eval_losses[-1]:.4f}")
    if len(eval_losses) > 1 and eval_losses[-1] > eval_losses[0]:
        print("  ⚠️  Val loss in aumento — overfitting rilevato")
        print("      Il miglior checkpoint è stato salvato automaticamente")
    else:
        print("  ✅ Val loss stabile o in discesa")

if train_losses and eval_losses:
    gap = abs(train_losses[-1] - eval_losses[-1])
    if gap > 1.0:
        print(f"\n  ⚠️  Gap train/val loss: {gap:.4f} — possibile overfitting")
    else:
        print(f"\n  ✅ Gap train/val loss: {gap:.4f} — generalizzazione buona")

# Salva modello finale
final_path = CHECKPOINT_DIR / "final"
trainer.save_model(str(final_path))
processor.save_pretrained(str(final_path))

print(f"\n✅ Training completato!")
print(f"   Modello salvato in: {final_path}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 Avvio training completo...
   Epoch:           3
   Esempi train:    2554
   Effective batch: 16
   Checkpoint ogni: fine epoch



Epoch,Training Loss,Validation Loss
1,0.828902,0.825426
2,0.705470,0.726947
3,0.610296,0.716030



 DIAGNOSTICA TRAINING

  Loss training — inizio: 2.0775  |  fine: 0.6103
  ✅ Loss scesa correttamente

  Loss validazione — inizio: 0.8254  |  fine: 0.7160
  ✅ Val loss stabile o in discesa

  ✅ Gap train/val loss: 0.1057 — generalizzazione buona

✅ Training completato!
   Modello salvato in: /home/maselli/develop/shield/outputs/checkpoints/final


---
## 7. Valutazione

Calcola ROUGE-1, ROUGE-2, ROUGE-L e BERTScore sul test set.
Salva anche un campione di output per revisione qualitativa manuale.

**Nota sul ROUGE con testo italiano**: il parametro `use_stemmer=True` usa
lo stemmer di Porter (pensato per l'inglese). Con testi italiani è preferibile
`use_stemmer=False` per evitare stemming scorretto su morfologia italiana.

**Nota sul BERTScore**: si specifica esplicitamente `model_type` per garantire
la riproducibilità indipendentemente dalla versione della libreria.
Si usa un modello multilingue (`xlm-roberta-large`) adatto ai referti in italiano.

**Baseline zero-shot**: la valutazione include la baseline del modello pre-fine-tuning
per quantificare il contributo effettivo dell'addestramento.

**Valori di riferimento dalla letteratura su IU X-Ray (testo in inglese):**

| Metrica   | Range tipico | Buono  |
|-----------|-------------|--------|
| ROUGE-1   | 0.30 – 0.45 | > 0.38 |
| ROUGE-2   | 0.12 – 0.22 | > 0.17 |
| ROUGE-L   | 0.27 – 0.38 | > 0.32 |

> Con testo italiano tradotto i valori assoluti potrebbero differire leggermente.
> L'importante è confrontare fine-tuned vs baseline zero-shot sullo stesso test set.

In [14]:
import torch
import json
import os
from PIL import Image
from tqdm.notebook import tqdm
from rouge_score import rouge_scorer as rs
from bert_score import score as bert_score_fn


def generate_report(image_path: str, max_new_tokens: int = 512) -> str:
    """
    Genera un referto per una singola immagine.
    Usa lo stesso system prompt del training per coerenza.
    """
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
            ]
        }
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    img    = Image.open(image_path).convert("RGB")
    device = next(model.parameters()).device
    inputs = processor(
        text=[text], images=[[img]], return_tensors="pt", padding=True
    ).to(device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
        )
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


def evaluate_on_test_set(label: str):
    """Valuta il modello corrente sul test set. Restituisce metriche e campioni qualitativi."""
    # test_data è già caricato come lista nella cella 9
    print(f"\n{'=' * 60}")
    print(f" VALUTAZIONE: {label}")
    print(f" Esempi nel test set: {len(test_data)}")
    print(f"{'=' * 60}")

    model.eval()
    predictions, references, qualitative = [], [], []

    # use_stemmer=False: lo stemmer di Porter è per l'inglese, non per l'italiano
    scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

    for i, example in enumerate(tqdm(test_data, desc=label)):
        msgs = example["messages"]
        image_path = next(
            (item["image"]
             for msg in msgs if msg["role"] == "user"
             for item in msg["content"]
             if isinstance(item, dict) and item.get("type") == "image"),
            None,
        )
        reference = next((m["content"] for m in msgs if m["role"] == "assistant"), "")

        if not image_path or not os.path.exists(image_path):
            continue
        try:
            prediction = generate_report(image_path)
        except Exception as e:
            print(f"Errore esempio {i}: {e}")
            continue

        predictions.append(prediction)
        references.append(reference)

        if i < 20:
            qualitative.append({
                "index": i, "image_path": image_path,
                "reference": reference, "prediction": prediction
            })

    # ROUGE
    rouge_acc = {"rouge1": [], "rouge2": [], "rougeL": []}
    for pred, ref in zip(predictions, references):
        r = scorer.score(ref, pred)
        for k in rouge_acc:
            rouge_acc[k].append(r[k].fmeasure)
    rouge_avg = {k: sum(v) / len(v) for k, v in rouge_acc.items()}

    # BERTScore — modello multilingue per testo italiano
    print("\nCalcolo BERTScore (può richiedere qualche minuto)...")
    P, R, F1 = bert_score_fn(
        predictions, references,
        model_type="xlm-roberta-large",
        lang="it",
        verbose=False,
    )
    bertscore_avg = {
        "precision": P.mean().item(),
        "recall":    R.mean().item(),
        "f1":        F1.mean().item(),
    }

    print(f"\n  Esempi valutati:  {len(predictions)}")
    print(f"  ROUGE-1:          {rouge_avg['rouge1']:.4f}")
    print(f"  ROUGE-2:          {rouge_avg['rouge2']:.4f}")
    print(f"  ROUGE-L:          {rouge_avg['rougeL']:.4f}")
    print(f"  BERTScore F1:     {bertscore_avg['f1']:.4f}")

    return {
        "label":        label,
        "rouge":        rouge_avg,
        "bertscore":    bertscore_avg,
        "num_examples": len(predictions),
        "qualitative":  qualitative,
    }


# ── Valutazione ───────────────────────────────────────────────
results_finetuned = evaluate_on_test_set("Fine-tuned QLoRA")

save_data = {k: v for k, v in results_finetuned.items() if k != "qualitative"}
with open(EVAL_DIR / "metrics_finetuned.json", "w") as f:
    json.dump(save_data, f, indent=2)
with open(EVAL_DIR / "qualitative_finetuned.json", "w", encoding="utf-8") as f:
    json.dump(results_finetuned["qualitative"], f, indent=2, ensure_ascii=False)

print(f"\n✅ Risultati salvati in: {EVAL_DIR}")



 VALUTAZIONE: Fine-tuned QLoRA
 Esempi nel test set: 320


Fine-tuned QLoRA:   0%|          | 0/320 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/maselli/develop/shield/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Calcolo BERTScore (può richiedere qualche minuto)...


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Esempi valutati:  320
  ROUGE-1:          0.3111
  ROUGE-2:          0.1128
  ROUGE-L:          0.2519
  BERTScore F1:     0.9009

✅ Risultati salvati in: /home/maselli/develop/shield/outputs/evaluation


In [15]:
# Mostra i primi 3 esempi qualitativi per revisione manuale
qualitative = results_finetuned["qualitative"]

print("=" * 60)
print(" REVISIONE QUALITATIVA — primi 3 esempi")
print("=" * 60)

for sample in qualitative[:3]:
    print(f"\n--- Esempio {sample['index']} ---")
    print(f"\nRIFERIMENTO (referto originale in italiano):")
    print(sample['reference'])
    print(f"\nGENERATO (output del modello):")
    print(sample['prediction'])
    print("-" * 40)

print("\nDomande da porsi durante la revisione:")
print("  - Il modello inventa patologie non presenti?")
print("  - Manca findings evidenti?")
print("  - Il linguaggio è clinicamente appropriato in italiano?")
print("  - Findings e impression (REPERTI/CONCLUSIONI) sono coerenti tra loro?")
print("  - La traduzione italiana è fluente o si vedono artefatti di traduzione automatica?")

 REVISIONE QUALITATIVA — primi 3 esempi

--- Esempio 0 ---

RIFERIMENTO (referto originale in italiano):
Reperti:
La siluetta cardiaca e le dimensioni del mediastino sono entro i limiti della norma. Non è presente edema polmonare. Non è presente consolidamento focale. Non sono presenti XXXX di versamento pleurico. Non vi è evidenza di pneumotorace.

Impressione:
Normale radiografia del torace x-XXXX.

GENERATO (output del modello):
Reperti:
I polmoni sono chiari bilateralmente. Nello specifico, nessuna evidenza di addensamento focale, pneumotorace o versamento pleurico. La siluetta cardio-mediastinica è normale. Le strutture ossee del torace visualizzate sono prive di alterazioni acute.

Impressione:
Nessuna alterazione cardiopolmonare acuta.
----------------------------------------

--- Esempio 1 ---

RIFERIMENTO (referto originale in italiano):
Reperti:
Le silhouette cardiaca e mediastinica sono nella norma. I polmoni sono ben espansi e chiari. Non sono presenti opacità focali dello 

---
## 8. Inferenza su singola immagine

In [ ]:
from IPython.display import display as ipy_display

# Modifica questo path con l'immagine che vuoi testare.
# Di default prende la prima .png disponibile in IMAGES_DIR.
TEST_IMAGE_PATH = str(list(IMAGES_DIR.glob("*.png"))[0])

print(f"Immagine di test: {TEST_IMAGE_PATH}")

img = Image.open(TEST_IMAGE_PATH)
ipy_display(img.resize((400, 400)))

print("\nGenerazione referto...")
report = generate_report(TEST_IMAGE_PATH)

print("\n" + "=" * 60)
print(" REFERTO GENERATO")
print("=" * 60)
print(report)
print("=" * 60)


---
## Note e troubleshooting

### OOM durante il training
Nella cella di configurazione (sezione 3), prova in ordine:
```python
BATCH_SIZE = 1                  # già al minimo
GRAD_ACCUMULATION_STEPS = 32    # aumenta questo invece
MAX_IMAGE_PIXELS = 501760       # dimezza la risoluzione
```

### Loss non scende
- Verifica la cella di verifica mascheratura (decodifica i token non mascherati)
- Controlla che il prompt utente sia identico tra training e inferenza
- Prova ad abbassare il learning rate: `LEARNING_RATE = 5e-5`

### Loss crolla a zero subito
Bug nella mascheratura del prompt nel data collator — esegui la cella di verifica
e controlla che il campo `Token risposta` mostri token che iniziano con `REPERTI:`.

### Il modello genera testo in inglese invece che italiano
- Verifica che il dataset tradotto sia quello effettivamente caricato
- Controlla `USER_PROMPT` — deve esplicitamente richiedere output in italiano

### Riprendere da checkpoint
```python
trainer.train(resume_from_checkpoint=True)
```

### Monitoraggio VRAM in tempo reale
```bash
watch -n 3 nvidia-smi
```